# 01 -- Data Exploration

Bank Account Fraud (BAF), Feedzai NeurIPS 2022, **Base** variant. This is
**account-opening** fraud (a fake/stolen identity opening a new account),
not card-transaction fraud: there is no transaction amount, no transaction
timestamp, and no account/customer id in the raw file. Every column is
available at *application time*.

This notebook runs the "first thirty minutes" checks that decide the rest of
the pipeline: split protocol, class balance, constant columns, sentinel
fractions, categorical cardinality, and the protected attribute for
fairness.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config

cfg = load_config(ROOT / "config.yaml")
pd.set_option("display.max_columns", 40)


In [2]:
from src.data_loader import load_raw

df = load_raw(cfg)
print(df.shape)
df.head()

(1000000, 32)


,fraud_bool,income,name_email_similarity,prev_address_months_count,current_address_months_count,customer_age,days_since_request,intended_balcon_amount,payment_type,zip_count_4w,velocity_6h,velocity_24h,velocity_4w,bank_branch_count_8w,date_of_birth_distinct_emails_4w,employment_status,credit_risk_score,email_is_free,housing_status,phone_home_valid,phone_mobile_valid,bank_months_count,has_other_cards,proposed_credit_limit,foreign_request,source,session_length_in_minutes,device_os,keep_alive_session,device_distinct_emails_8w,device_fraud_count,month
0,0,0.3,0.986506,-1,25,40,0.006735,102.453711,AA,1059,13096.035018,7850.955007,6742.080561,5,5,CB,163,1,BC,0,1,9,0,1500.0,0,INTERNET,16.224843,linux,1,1,0,0
1,0,0.8,0.617426,-1,89,20,0.010095,-0.849551,AD,1658,9223.283431,5745.251481,5941.664859,3,18,CA,154,1,BC,1,1,2,0,1500.0,0,INTERNET,3.363854,other,1,1,0,0
2,0,0.8,0.996707,9,14,40,0.012316,-1.490386,AB,1095,4471.472149,5471.988958,5992.555113,15,11,CA,89,1,BC,0,1,30,0,200.0,0,INTERNET,22.730559,windows,0,1,0,0
3,0,0.6,0.475100,11,14,30,0.006991,-1.863101,AB,3483,14431.993621,6755.344479,5970.336831,11,13,CA,90,1,BC,0,1,1,0,200.0,0,INTERNET,15.215816,linux,1,1,0,0
4,0,0.9,0.842307,-1,29,40,5.742626,47.152498,AA,2339,7601.511579,5124.046930,5940.734212,1,6,CA,91,0,BC,1,1,26,0,200.0,0,INTERNET,3.743048,other,0,1,0,0


## Class balance -- the number every other metric has to be read against

In [3]:
fraud_rate = df[cfg.data.target_col].mean()
print(f"Fraud rate: {fraud_rate:.4%}")
print(f"Accuracy of an all-zero (never predict fraud) classifier: {1 - fraud_rate:.4%}")
print("-> accuracy is meaningless on this data; PR-AUC / TPR@5%FPR are used instead.")

Fraud rate: 1.1029%
Accuracy of an all-zero (never predict fraud) classifier: 98.8971%
-> accuracy is meaningless on this data; PR-AUC / TPR@5%FPR are used instead.


## Is the split random or temporal? (decides whether `month` is a feature)

VERIFIED (see README.md): this kit uses a stratified RANDOM 70/15/15 split, not the NeurIPS paper's temporal protocol, so `month` is kept as an ordinary feature. The check below is the same one that led to that decision.

In [4]:
print(df["month"].value_counts().sort_index())
print()
print(df.groupby("month")[cfg.data.target_col].agg(["mean", "size"]))

month
0    132440
1    127620
2    136979
3    150936
4    127691
5    119323
6    108168
7     96843
Name: count, dtype: int64

           mean    size
month                  
0      0.011326  132440
1      0.009387  127620
2      0.008746  136979
3      0.009222  150936
4      0.011371  127691
5      0.011825  119323
6      0.013405  108168
7      0.014746   96843


## Constant columns

In [5]:
nun = df.nunique().sort_values()
print(nun.head(10))
print()
print("device_fraud_count unique values:", df['device_fraud_count'].unique())

device_fraud_count           1
fraud_bool                   2
keep_alive_session           2
foreign_request              2
phone_home_valid             2
has_other_cards              2
email_is_free                2
source                       2
phone_mobile_valid           2
device_distinct_emails_8w    4
dtype: int64

device_fraud_count unique values: [0]


## Sentinel (-1 = missing) columns vs. legitimate-negative columns

Six columns use `-1` as a missing sentinel. Two other columns (`credit_risk_score`, `velocity_6h`) have real negative values and must NOT be treated as missing -- see 01-DATASET-BIBLE.md in the sibling kit.

In [6]:
for c in cfg.sentinel_cols:
    print(f"{c:35s} frac negative: {(df[c] < 0).mean():.4f}")
print()
for c in cfg.legitimate_negative_cols:
    print(f"{c:35s} frac negative (LEGITIMATE, not missing): {(df[c] < 0).mean():.4f}")

prev_address_months_count           frac negative: 0.7129
current_address_months_count        frac negative: 0.0043
bank_months_count                   frac negative: 0.2536
session_length_in_minutes           frac negative: 0.0020
device_distinct_emails_8w           frac negative: 0.0004
intended_balcon_amount              frac negative: 0.7425

credit_risk_score                   frac negative (LEGITIMATE, not missing): 0.0144
velocity_6h                         frac negative (LEGITIMATE, not missing): 0.0000


## Categorical columns

In [7]:
for c in cfg.categorical_cols:
    print(c, df[c].nunique(), df[c].unique())

payment_type 5 <ArrowStringArray>
['AA', 'AD', 'AB', 'AC', 'AE']
Length: 5, dtype: str
employment_status 7 <ArrowStringArray>
['CB', 'CA', 'CC', 'CF', 'CD', 'CE', 'CG']
Length: 7, dtype: str


housing_status 7 <ArrowStringArray>
['BC', 'BE', 'BD', 'BA', 'BB', 'BF', 'BG']
Length: 7, dtype: str
source 2 <ArrowStringArray>
['INTERNET', 'TELEAPP']
Length: 2, dtype: str


device_os 5 <ArrowStringArray>
['linux', 'other', 'windows', 'x11', 'macintosh']
Length: 5, dtype: str


## Protected attribute for fairness: `customer_age > 50`

Ages are rounded to the decade (9 distinct values), so the BAF paper's strictly-greater-than-50 cut matters: `>= 50` would silently move an entire bucket of applicants across the line.

In [8]:
print(df["customer_age"].value_counts().sort_index())
older = df["customer_age"] > cfg.protected_attribute.threshold
print()
print("Fraud rate, age > 50:", df.loc[older, cfg.data.target_col].mean())
print("Fraud rate, age <= 50:", df.loc[~older, cfg.data.target_col].mean())

customer_age
10     20987
20    245855
30    311433
40    238712
50    140353
60     34770
70      6517
80      1297
90        76
Name: count, dtype: int64

Fraud rate, age > 50: 0.034692920768870136
Fraud rate, age <= 50: 0.009974512712307017


## No identifier and no leakage columns

There is no `account_id`/`customer_id` in the raw file (no identifier-exclusion step was needed) and no post-decision outcome columns (chargeback status, investigation result, etc.) -- every feature is available at application time.

In [9]:
from src.data_validation import check_no_leakage
import re
id_pattern = re.compile(r"(^id$|_id$|^id_|account_id|customer_id)", re.IGNORECASE)
print("identifier-like columns:", [c for c in df.columns if id_pattern.search(c)])
print("leakage-like columns:", check_no_leakage(df))

identifier-like columns: []
leakage-like columns: []


## What was explicitly skipped, and why

This dataset has no transaction amount, no transaction timestamp (only a
coarse 0-7 `month` index), and no transaction history, so the following
spec items do not apply and were skipped rather than faked:

- `amount_log`, `transactions_per_hour` -- no transaction amount column exists.
- `hour`, `day_of_week`, `is_weekend` -- no real timestamp, only `month` (0-7).
- `current_amount / historical_average` deviation features -- no transaction
  history to compute a historical average from.

See `src/feature_engineering.py::SKIPPED_FEATURES` for the same list kept
next to the code.